# fastllm

> Unified async LLM API — swap providers without changing code

`fastllm` gives you a single `acomplete` function that works identically across **Anthropic**, **OpenAI** (Responses + Chat), **Gemini**, and OpenAI-compatible providers like **Kimi**. Define your messages and tools once in a canonical format, then switch models by changing one string.

## Install

Clone and install locally into your `aai-ws` env

In [ ]:
#| hide
from fastllm.types import *
from fastllm.normalize import *
from fastllm.streaming import *
from fastllm.acomplete import *

In [ ]:
#| hide
from cachy.core import enable_cachy, disable_cachy, doms

In [ ]:
#| hide
enable_cachy(doms=(*doms,'api.moonshot.ai'), hdrs=('content-type',))

## Setup

In [ ]:
from fastllm.acomplete import acomplete, mk_tool_res_msg
from fastllm.types import Msg, Part, PartType
from fastllm.normalize import Completion
import asyncio, json

# Helpers
def user(text): return Msg(role='user', content=[Part(type=PartType.text, text=text)])

async def stream(msgs, model, **kw):
    "Stream a response, printing text/thinking as it arrives. Returns the final Completion."
    cnt, max_think = 0, 10
    async for o in await acomplete(msgs, model, stream=True, **kw):
        if not isinstance(o, Completion):
            if o.get('thinking') and cnt < max_think: print('🧠', end='', flush=True)
            if txt := o.get('text'): print(txt, end='', flush=True)
            cnt += 1
    print()
    return o

In [ ]:
mtok = 1024

## Chat — One Interface, Every Provider

The same `acomplete` call works with Claude, GPT, Gemini, and Kimi. Just change the model name:

In [ ]:
models = [
    ('claude-sonnet-4-20250514', {}),
    ('gpt-4o-mini', {}),
    ('models/gemini-3-flash-preview', {}),
    ('kimi-k2.5', dict(third_party_name='moonshot')),
]
for name, kw in models:
    r = await acomplete([user("Say 'hello' in French.")], model=name, max_tokens=mtok, **kw)
    print(f"{name:>30s} → {r.message.content[0].text.strip()}")

      claude-sonnet-4-20250514 → Bonjour
                   gpt-4o-mini → "Hello" in French is "Bonjour."
 models/gemini-3-flash-preview → Bonjour OR Salut (informal)


                     kimi-k2.5 → The user is asking me to say "hello" in French. This is a very simple request. The French word for "hello" is "bonjour". 

I should provide a direct, helpful response. I can also add a bit of context like pronunciation or usage if helpful, but the core answer is simply "bonjour".

Let me provide the answer clearly.


## Multi-Turn — Swap Providers Mid-Conversation

Build a conversation with one provider, then seamlessly continue it with another. `fastllm` translates between every provider's native format automatically:

In [ ]:
# Turn 1: Claude starts the conversation
msgs = [user("Name the 3 largest planets in our solar system. One sentence.")]
print("Claude: ", end='')
r1 = await stream(msgs, model='claude-sonnet-4-20250514', max_tokens=mtok)

Claude: The three largest planets in our solar system

 are Jupiter, Saturn, and Neptune.

In [ ]:
# Turn 2: Switch to GPT — just change the model string
msgs += [r1.message, user("Which one has the most moons?")]
print("GPT: ", end='')
r2 = await stream(msgs, model='gpt-4o-mini', max_tokens=mtok)

GPT: As

 of

 now

,

 Saturn

...

,

 surpass

ing

 Jupiter

.

In [ ]:
# Turn 3: Switch to Gemini — same msgs, different provider
msgs += [r2.message, user("Summarize our conversation in one sentence.")]
print("Gemini: ", end='')
r3 = await stream(msgs, model='models/gemini-3-flash-preview', max_tokens=mtok)

Gemini: The conversation identified the three largest planets

 in the solar system and established that Saturn currently has the most moons.

In [ ]:
# Turn 4: Switch to Kimi — works the same way
msgs += [r3.message, user("Thanks! What's one surprising fact about Saturn?")]
print("Kimi: ", end='')
r4 = await stream(msgs, model='kimi-k2.5', max_tokens=mtok, third_party_name='moonshot')

Kimi: 

🧠

🧠

🧠

🧠

...

 enough

 to

 hold

 it

.

## System Prompts

Pass a system prompt to any provider — `fastllm` maps it to each API's native mechanism (Anthropic `system`, OpenAI `instructions`, Gemini `system_instruction`):

In [ ]:
sys = "You are a pirate chef. Always respond in pirate speak and mention food."

print("Claude: ", end='')
r = await stream([user("What should I do today?")], model='claude-sonnet-4-20250514', system=sys, max_tokens=mtok)

print("Gemini: ", end='')
r = await stream([user("What should I do today?")], model='models/gemini-3-flash-preview', system=sys, max_tokens=mtok)

Claude: 

Ahoy there, me

 hearty! *tips chef's hat with a feather*

Why, ye should

 be settin' sail for the galley, of course! Here be some fine suggestions for 

yer day's adventures:

**In the Kitchen:**
- Whip up a hearty batch of ship

...

 onions ye managed to swipe from the captain’s pantry. Make sure ye season it with enough black pepper to make a mermaid

 sneeze!

And when the sun dips below the horizon, crack open a coconut, splash in a bit o' rum, and feast

 like a king o' the high seas. Just don't let the ship's cat near yer plate, or ye

'll be fightin' for every scrap o' that grilled red snapper!

Now move yer barnacle-covered

 boots and get to cookin'! Arrr!

## Tool Calling — Define Once, Use Anywhere

Define tools in a single canonical format. `fastllm` translates to each provider's native tool schema automatically. Here we start a tool-use conversation with Claude, provide the result, then switch to GPT and Gemini to continue:

In [ ]:
tools = [{"type": "function", "function": {
    "name": "get_weather",
    "description": "Get current weather for a city",
    "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}
}}]

# Turn 1: Claude requests the tool
msgs = [user("What's the weather in Paris?")]
r1 = await stream(msgs, model='claude-sonnet-4-20250514', tools=tools, max_tokens=mtok)
print("Tool calls:", r1.tool_calls)

I'll check the current

 weather in Paris for you.


Tool calls: [ToolCall(id='toolu_01TnN16XYqhDH8KHEJytrJPF', name='get_weather', arguments={'city': 'Paris'}, server=False, extra={'caller': {'type': 'direct'}})]


In [ ]:
# Provide the tool result
tool_parts = [p for p in r1.message.content if p.type == PartType.tool_use]
msgs += [r1.message, mk_tool_res_msg(tool_parts, ['22°C, sunny with light clouds'])]

# Turn 2: Switch to GPT to interpret the result
msgs.append(user("Should I bring a jacket?"))
print("GPT: ", end='')
r2 = await stream(msgs, model='gpt-4o-mini', tools=tools, max_tokens=mtok)

GPT: 

With

 the

 current

 temperature

...

 cold

 in

 the

 evening

.

In [ ]:
# Turn 3: Gemini sees the full cross-provider tool history
msgs += [r2.message, user("How about tomorrow — will it rain?")]
r3 = await stream(msgs, model='models/gemini-3-flash-preview', tools=tools, max_tokens=mtok)
print("Tool calls:", r3.tool_calls)

I'll check the weather forecast for Paris tomorrow to see if rain is expected.




Tool calls: [ToolCall(id='o6g6wxhb', name='get_weather', arguments={'city': 'Paris tomorrow'}, server=False, extra={'thoughtSignature': 'ErcjCrQjAQw51sfDB5AYrjjKKMWTwQ96Ayy+SWBt6dBSO++xia5Bn35mTfyUraFWSyQrvsmwZT2onkHwL3xXpvvQLrhORAIRn6MTu9amZgvdUFNvDqs8VX2wvVrA7pC5HftZz4HzMrQLMxwTR/a5qleYeZzBChZeLSaiBkuhmi7OKwik1x443HttqyqUsXap7hkQiLK/BjHp7LEatKDhZoZqKbWrspClGqQEZeAewQxd/fe3CS12tzF6Dd1ChCBIevh2US9gYVPVqAC3ZgKyLGYD6jT2ti8Unaqo1/+4UPG0/gUjHH4qLqaguXjvnSxCjnn8u42NUGOtPWiRQH87iJ9TcGP8idG/RrmcsTHEifbibMAP3HqnrbsO5+J7ZyH0nHc5fbF3KqlohUVBAbqDWhTqGNVaax6gw9xQBdmaCggyDb7VtiLbNkZShPU4iCkaGZJiLkPM9cG2uMy1zKNjftrII9WfEZb7nyFXTDmbqD0kqQhFmjKk5a2MslYfKOL8Y7F4VSdJMwXwtb0XL5ylsv1i08p1zCmgzvUc7YvvP+MJOMiGwYir/AEqaINyR9tf16fGKJKtPR/jCgIKdu5mN68HXFSztIKTAINm92Ezq142n6fEgKsSZRHsZGfV0H5atww8zju5YKTIAY9odeFDxv++iBb33EKcMY9dTe6qBdi27MvSejJu4b2ODsUJpqW96Wpm9sp4Vk5XxSvlGxx6nEvrpgzHs/U2m1x2B9zXuo+qFsm+DRZN1NQSbqvuERjMjYRvTQFJi1MhudJ6ETixyRd9YUyTSwi3Ntx+6Vt6iIdtB9yiNNzkM30NNdJNPepO3CMBN5udxoHJz7Cg0f2lScYQR9MBp2

## Tool Choice

Control whether the model must use tools, can't use tools, or decides on its own:

In [ ]:
# Force the model to call a tool (even for a greeting)
r = await stream([user("Hello there!")], model='claude-sonnet-4-20250514', tools=tools, tool_choice='required', max_tokens=mtok)
print("Forced:", r.tool_calls)

# Prevent tool use (model must answer directly)
print("\nNo tools: ", end='')
r = await stream([user("What's the weather?")], model='claude-sonnet-4-20250514', tools=tools, tool_choice='none', max_tokens=mtok)


Forced: [ToolCall(id='toolu_01KXyFbVrdSG3JSNMwpveupZ', name='get_weather', arguments={'city': '<UNKNOWN>'}, server=False, extra={'caller': {'type': 'direct'}})]

No tools: 

I'd

 be happy to help you check the weather! However, I need to know which city you'd like me to check the weather for. Could you please tell me the city name?

## Thinking / Extended Reasoning

Enable model reasoning with `reasoning_effort`. The canonical values (`low`, `medium`, `high`) map to each provider's native budget system. Thinking tokens stream as 🧠:

In [ ]:
# Claude with thinking
print("Claude: ", end='')
r = await stream([user("What is 127 × 849?")], model='claude-sonnet-4-20250514', reasoning_effort='low', max_tokens=8192)
for p in r.message.content:
    if p.type == PartType.thinking: print(f"\n🧠 {p.text[:150]}...")

# Kimi with thinking — same interface
print("\nKimi: ", end='')
r = await stream([user("What is 127 × 849?")], model='kimi-k2.5', reasoning_effort='low', max_tokens=8192, third_party_name='moonshot')
for p in r.message.content:
    if p.type == PartType.thinking: print(f"\n🧠 {p.text[:150]}...")

Claude: 🧠

🧠

🧠

🧠

🧠

...

}

823

$

**



🧠 This is a straightforward multiplication problem. I need to calculate 127 × 849.

Let me break this down:
127 × 849

I can use the distributive proper...


## Web Search (Server Tools)

OpenAI's Responses API supports server-side web search. Server tool calls are normalized alongside regular tool calls:

In [ ]:
ws_tools = [{"type": "web_search_preview"}]
print("GPT + web search: ", end='')
r = await stream([user("What is the latest Python release?")], model='gpt-4o-mini', tools=ws_tools, max_tokens=512)
print(f"\nServer tools used: {[tc.name for tc in r.tool_calls if tc.server]}")

GPT + web search: 

As

 of

 April

16

...

 in

 this

 version

.



Server tools used: ['web_search']


## Caching (Anthropic)

Anthropic supports prompt caching via `cache_control`. Pass it through `Part.data` — repeat calls with the same cached content save tokens:

In [ ]:
# Cache a long system prompt (must be >1024 tokens for Anthropic caching)
long_ctx = "You are an expert on the solar system. " * 200
system = Part(type=PartType.text, text=long_ctx, data={'cache_control': {'type': 'ephemeral'}})

print("Call 1: ", end='')
r1 = await stream([user("What is Jupiter's mass?")], model='claude-sonnet-4-20250514', system=system, max_tokens=mtok)
print(f"Usage: {r1.usage}")

print("Call 2: ", end='')
r2 = await stream([user("How about Saturn?")], model='claude-sonnet-4-20250514', system=system, max_tokens=mtok)
print(f"Usage: {r2.usage}")

Call 1: 

Jupiter

's mass is approximately 1.898 × 10²⁷ kilograms (or 1,898,

000,000,000,000,000,000,000,000 kg).

To put this in perspective:
- Jupiter is about 318

 times more massive than Earth
- It contains more than twice the mass of all other planets in our

...

 moon with geysers of water ice erupting from its south pole

**Other Notable Features:**
- A day on Saturn l

asts about 10.7 hours
- It takes about 29.5 Earth years to orbit the Sun
- Has hexagonal storm patterns at

 its north pole
- Wind speeds can reach up to 1,100 mph (1,800 km/h)

What aspect of Saturn interests

 you most?


Usage: Usage(prompt_tokens=10, completion_tokens=359, total_tokens=369, raw={'input_tokens': 10, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 1802, 'output_tokens': 359})


## Media Inputs

Send images to any provider that supports them. The canonical `input_image` part works everywhere:

In [ ]:
img_url = "https://img.freepik.com/free-photo/mountain-range-body-water_53876-139760.jpg?semt=ais_hybrid&w=740&q=80"
img_msg = Msg(role='user', content=[
    Part(type=PartType.input_image, text=img_url),
    Part(type=PartType.text, text="What do you see in this image?")
])

for name, kw in [('claude-sonnet-4-20250514', {}), ('gpt-4o-mini', {}), ('models/gemini-3-flash-preview', {})]:
    print(f"{name:>30s}: ", end='')
    r = await stream([img_msg], model=name, max_tokens=mtok, **kw)

      claude-sonnet-4-20250514: 

I see a beautiful

, serene landscape with a calm lake that creates perfect mirror reflections. In the foreground, there's

 a wooden deck or pier made of weathered planks. The lake

 reflects the surrounding scenery almost like glass - showing

...

 of evergreen trees, appearing in various shades of deep green and dark silhouette.
*   **Background:** Towering above

 the trees are **massive, rugged mountains**. The closer peaks are shrouded in a hazy blue mist, while the taller, more

 distant peaks are covered in bright white snow and glaciers.
*   **Sky:** The sky is a pale, washed

-out blue, with soft, white clouds clinging to the highest mountain peaks. 

The overall atmosphere of the image is quiet

, cold, and majestic.

`fastllm` supports four media part types via `PartType`. Provider support varies:

| Part Type | Anthropic | OpenAI Responses | OpenAI Chat | Gemini |
|---|---|---|---|---|
| `input_image` | ✅ | ✅ | ✅ | ✅ |
| `input_audio` | ❌ | ❌ (coming soon) | ✅ | ✅ |
| `input_video` | ❌ | ❌ | ❌ | ✅ |
| `input_file` | ✅ | ✅ | ✅ | ✅ |

All media parts accept either a URL or a base64 data URL in `Part.text`. Unsupported combinations raise a clear `ValueError`.